In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

# 10 — Setup Environment Skenario B

Target: **PC kampus** (RTX 4090 / i9-14900K / 32 GB, Jupyter via Tailscale). Jalankan SEKALI.

Dua env conda:
- `s2-main` (python 3.10): ultralytics (deteksi), TrackEval, OC-SORT runner, plot.
- `s2-diffmot` (python 3.9): DiffMOT — **torch 2.0.1 WAJIB dari index cu118** (wheel PyPI 2.0.1 rusak: hilang dependensi nvidia → `libnvrtc not found`).

Referensi: `docs/plans/2026-08-02-phase9-skenario-b-tracker.md`


In [ ]:
!nvidia-smi

In [ ]:
!conda --version

### Env s2-main

In [ ]:
!conda create -n s2-main python=3.10 -y

In [ ]:
!conda run -n s2-main pip install -q ipykernel ultralytics trackeval motmetrics filterpy loguru numpy pandas openpyxl matplotlib huggingface_hub

### Clone repo pihak ketiga (sekali)

In [ ]:
!git clone --depth 1 https://github.com/Kroery/DiffMOT $S2_EXT/diffmot
!git clone --depth 1 https://github.com/noahcao/OC_SORT $S2_EXT/OC_SORT
!git clone --depth 1 https://github.com/JonathonLuiten/TrackEval $S2_EXT/TrackEval

### Env s2-diffmot

`torch==2.0.1` dari **index cu118**, BUKAN PyPI. `cython-bbox` rawan gagal compile; bila
gagal: `conda install -c conda-forge cython-bbox` lalu lanjutkan.

In [ ]:
!conda create -n s2-diffmot python=3.9 -y

In [ ]:
!conda run -n s2-diffmot pip install -q ipykernel torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!conda run -n s2-diffmot pip install -r $S2_EXT/diffmot/requirement.txt

### External deps DiffMOT (YOLOX, deep-person-reid, fast_reid)

In [ ]:
!cd $S2_EXT/diffmot/external/YOLOX && conda run -n s2-diffmot pip install -r requirements.txt && conda run -n s2-diffmot python setup.py develop

In [ ]:
!cd $S2_EXT/diffmot/external/deep-person-reid && conda run -n s2-diffmot pip install -r requirements.txt && conda run -n s2-diffmot python setup.py develop

In [ ]:
!cd $S2_EXT/diffmot/external/fast_reid && conda run -n s2-diffmot pip install -r docs/requirements.txt

### Daftarkan kernel Jupyter

In [ ]:
!conda run -n s2-main python -m ipykernel install --user --name s2-main
!conda run -n s2-diffmot python -m ipykernel install --user --name s2-diffmot

### Bobot DiffMOT (release v1.0)

- Motion D²MP: `MOT_epoch800.pt` (MOT17+MOT20) → rename `mot_epoch800.pt`; `DanceTrack_epoch800.pt` → rename `dancetrack_epoch800.pt`
- ReID (FastReID): `mot20_sbs_S50.pth`, `dance_sbs_S50.pth`

Posisi checkpoint motion = `{diffmot}/experiments/{eval_expname}/{dataset}_epoch{epoch}.pt`
(konvensi `diffmot.py::_build_model` — `dataset` = arg `--dataset`).

In [ ]:
!mkdir -p $S2_EXT/diffmot/experiments/diffmot_mot $S2_EXT/diffmot/experiments/diffmot_dance $S2_EXT/diffmot/external/weights

In [ ]:
!wget -qO $S2_EXT/diffmot/experiments/diffmot_mot/mot_epoch800.pt https://github.com/Kroery/DiffMOT/releases/download/v1.0/MOT_epoch800.pt
!wget -qO $S2_EXT/diffmot/experiments/diffmot_dance/dancetrack_epoch800.pt https://github.com/Kroery/DiffMOT/releases/download/v1.0/DanceTrack_epoch800.pt
!wget -qO $S2_EXT/diffmot/external/weights/mot20_sbs_S50.pth https://github.com/Kroery/DiffMOT/releases/download/v1.0/mot20_sbs_S50.pth
!wget -qO $S2_EXT/diffmot/external/weights/dance_sbs_S50.pth https://github.com/Kroery/DiffMOT/releases/download/v1.0/dance_sbs_S50.pth

### Verifikasi

In [ ]:
!ls -lh $S2_EXT/diffmot/experiments/diffmot_mot/ $S2_EXT/diffmot/experiments/diffmot_dance/ $S2_EXT/diffmot/external/weights/

In [ ]:
!conda run -n s2-diffmot python -c "import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))"

In [ ]:
!conda run -n s2-main python -c "import torch, ultralytics, trackeval, motmetrics; print('s2-main OK | torch', torch.__version__, '| ultralytics', ultralytics.__version__)"

**Lanjut**: notebook `20_s2_download_data.ipynb` (kernel `s2-main`).